# NB 2.2 &mdash; Regressió lineal múltiple: afegir variables d'una en una

**MP 5134** &mdash; UT2

*Dades: AEMET, estació de l'aeroport de Palma.*

---
### Què farem avui

Al NB 2.1 vam predir la massa d'un pingüí amb una sola mesura. Avui tornem a les
dades meteorològiques de Palma i ens fem una pregunta molt pràctica: **si li donem
més variables al model, prediu millor?** I si és així, quant millor?

Abans, però, farem una cosa que només es fa una vegada en tot el curs: **segellar
el test**.

Quan acabis hauries de saber respondre aquestes preguntes:

- Què és el test segellat i per què no el podem tocar fins a la UT10?
- Per què necessitem un conjunt de *validació* a més del d'entrenament?
- Afegir variables sempre ajuda? Quant?
- Com es llegeixen els coeficients d'un model amb moltes variables, i quins
  paranys tenen?

## 1. Les dades

Carreguem el fitxer meteorològic de la UT1. Cada fila és un dia i volem predir
`tmax_dema`, la temperatura màxima **de l'endemà**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Dades diàries de l'estació B278 (aeroport de Palma), publicades per l'AEMET
URL_DADES = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/disseny-avaluacio-models-ml/refs/heads/main/UT01-Entorn_de_treball_primer_model/aemet/meteo_palma.csv"

df = pd.read_csv(URL_DADES, parse_dates=["fecha"])

print(f"Dies al fitxer: {len(df)}, del {df['fecha'].min():%d/%m/%Y} al {df['fecha'].max():%d/%m/%Y}")
df.head()

Onze anys de dades, dia a dia. Recorda del NB 1.1 què és cada columna: les
temperatures (`tmed`, `tmin`, `tmax`), la pluja (`prec`), el vent (`velmedia`,
`racha`), les hores de sol (`sol`) i la pressió atmosfèrica (`presMax`,
`presMin`). Les columnes que acaben en `_dema` són del dia següent, i per això són
**respostes**, no entrades.

## 2. El test segellat

Aquesta secció és curta però és de les més importants del curs.

Durant les pròximes unitats entrenarem desenes de models sobre aquestes dades:
regressions, arbres, boscos, SVM... I cada vegada que en provem un i mirem el
resultat, prendrem decisions: aquesta variable sí, aquesta no, aquest model és
millor que aquell. Cada decisió que prenem mirant uns resultats **adapta una mica
el nostre treball a aquelles dades concretes**.

Si sempre mirem el mateix conjunt de prova, al cap de tres mesos haurem ajustat
tant les decisions a aquells dies que el número que en traguem ja no serà una
estimació honesta de com funcionarà el model amb dies nous. És una trampa que no
es veu, perquè cap decisió sola és una trampa: és la suma de totes.

La solució és apartar ara un tros de les dades i **no mirar-lo mai** fins al
final. L'obrirem tots junts a classe a la **UT10**, amb el projecte, i serà
l'única manera de saber de veritat si tot el que hem après funciona.

Farem servir com a test segellat **els dos últims anys, 2024 i 2025**.

In [ ]:
TALL_SEGELLAT = "2024-01-01"

test_segellat = df[df["fecha"] >= TALL_SEGELLAT]
dades = df[df["fecha"] < TALL_SEGELLAT].copy()

print(f"Dies que ens quedem per treballar: {len(dades)}")
print(f"Dies del test segellat:            {len(test_segellat)}  ({100 * len(test_segellat) / len(df):.0f} % del total)")

del test_segellat   # des d'aquí, en aquest notebook ja no existeix

Un 18% de les dades, dins del marge habitual d'entre el 10% i el 30%. La línia
`del test_segellat` esborra la variable: és un gest simbòlic, però fa que sigui
impossible fer-la servir per descuit més avall.

**A partir d'ara, tots els notebooks del curs que facin servir aquestes dades
començaran amb aquestes mateixes línies.** És la manera de garantir que el segell
no es trenca.

Potser et preguntes per què no hem fet servir `train_test_split`, com a la UT1,
que tria els dies a l'atzar. Hi ha dos motius:

- **Volem predir el futur.** Un model meteorològic s'entrena amb el passat i es fa
  servir amb dies que encara no han arribat. Separar per dates imita aquesta
  situació.
- **Els dies seguits s'assemblen molt.** Si el 14 de juliol queda a l'entrenament
  i el 15 a la prova, el model té ajuda: ha vist un dia gairebé idèntic. Tallant
  per dates, el model no pot fer servir aquesta drecera.

## 3. Entrenament i validació

Amb el test segellat, tenim un problema: **sobre quines dades comparem els models
d'avui?** No podem fer servir el test, i avaluar sobre l'entrenament no serveix,
com ja saps.

La sortida és tornar a partir el que ens queda. Els anys 2015 a 2021 seran per
entrenar, i el 2022 i el 2023 seran per **validar**: per comparar models i decidir.

In [ ]:
FEATURES = ["tmax", "tmin", "tmed", "sol", "presMax", "presMin",
            "prec", "velmedia", "mes", "dia_any"]
OBJECTIU = "tmax_dema"

dades = dades.dropna(subset=FEATURES + [OBJECTIU])

TALL_VALIDACIO = "2022-01-01"
train = dades[dades["fecha"] < TALL_VALIDACIO]
valid = dades[dades["fecha"] >= TALL_VALIDACIO]

print(f"Dies d'entrenament (2015-2021): {len(train)}")
print(f"Dies de validació  (2022-2023): {len(valid)}")

Ara tenim les dades en tres trossos, i convé tenir clar per a què serveix cadascun:

| Tros | Anys | Per a què |
|---|---|---|
| **Entrenament** | 2015-2021 | el model hi aprèn amb `fit()` |
| **Validació** | 2022-2023 | nosaltres hi comparem models i decidim |
| **Test segellat** | 2024-2025 | no es toca fins a la UT10 |

Partir en tres trossos una sola vegada és la versió senzilla. A la **UT7**
veurem la validació creuada, que fa aquesta mateixa idea molt més sòlida.

El `dropna()` treu els pocs dies on falta alguna mesura, perquè `fit()` no accepta
valors absents.

## 4. Abans de res, dos models de referència

Ja saps que un número tot sol no vol dir res. Avui en farem servir dos, de
referències:

- **Dir sempre la mitjana.** El `DummyRegressor` de la UT1.
- **Dir que demà farà la mateixa màxima que avui.** És el que diria qualsevol
  persona sense saber res d'aprenentatge automàtic, i en meteorologia té nom
  propi: la *persistència*.

Primer preparem una funció que calculi les mètriques del NB 2.1, perquè les
farem servir moltes vegades.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def mesurar(y_real, y_predit):
    return {
        "MAE (°C)": mean_absolute_error(y_real, y_predit),
        "RMSE (°C)": np.sqrt(mean_squared_error(y_real, y_predit)),
        "R2": r2_score(y_real, y_predit),
    }

In [ ]:
from sklearn.dummy import DummyRegressor

mitjana = DummyRegressor(strategy="mean").fit(train[["tmax"]], train[OBJECTIU])

referencies = pd.DataFrame({
    "dir sempre la mitjana": mesurar(valid[OBJECTIU], mitjana.predict(valid[["tmax"]])),
    "demà igual que avui": mesurar(valid[OBJECTIU], valid["tmax"]),
}).T

referencies.round(2)

Mira la diferència entre les dues referències. Dir sempre la mitjana s'equivoca
gairebé **6 °C** de mitjana: no té en compte ni si som a l'hivern o a l'estiu.
Dir que demà farà com avui s'equivoca **1,6 °C** i ja té un R2 de 0,89.

Per a nosaltres, **la persistència és el llistó de veritat**. Guanyar el model que
diu la mitjana és fàcil; si el nostre model no guanya el sentit comú, no val la
pena.

## 5. Una sola variable: la màxima d'avui

Comencem com al NB 2.1, amb una recta. L'entrada serà `tmax`, la màxima d'avui.

In [ ]:
from sklearn.linear_model import LinearRegression

model_1 = LinearRegression().fit(train[["tmax"]], train[OBJECTIU])

print(f"Pendent:             {model_1.coef_[0]:.3f}")
print(f"Ordenada a l'origen: {model_1.intercept_:.2f} °C")
print()
print("Sobre la validació:")
for nom, valor in mesurar(valid[OBJECTIU], model_1.predict(valid[["tmax"]])).items():
    print(f"  {nom:>9}: {valor:.3f}")

Llegim el model en veu alta, com vam fer amb els pingüins:

> **La màxima de demà serà el 94% de la màxima d'avui, més 1,4 °C.**

És gairebé la persistència. La diferència és que el model ha après que els dies
molt calorosos solen anar seguits d'un dia una mica menys calorós, i els dies
freds, d'un dia una mica menys fred: estira les prediccions cap al centre.

I el resultat? Un MAE de 1,61 °C contra el 1,64 °C de la persistència. **Guanyem,
però per tres centèsimes de grau.** Val la pena que t'hi aturis, perquè és una
situació molt real: de vegades el model entrenat amb tot el rigor només millora
una mica el que ja diria el sentit comú.

In [ ]:
temperatures = pd.DataFrame({"tmax": np.linspace(5, 40, 50)})

plt.figure(figsize=(7, 5))
plt.scatter(valid["tmax"], valid[OBJECTIU], alpha=0.2, s=10, label="dies de validació")
plt.plot(temperatures["tmax"], model_1.predict(temperatures), color="tab:red", linewidth=2, label="recta apresa")
plt.plot(temperatures["tmax"], temperatures["tmax"], color="gray", linestyle="--", label="demà igual que avui")
plt.xlabel("Màxima d'avui (°C)")
plt.ylabel("Màxima de demà (°C)")
plt.legend()
plt.show()

La recta vermella i la línia grisa gairebé es tapen: a la pràctica, el model ha
après la persistència. El núvol és molt més estret que el dels pingüins, i per
això el R2 és tan alt.

## 6. Afegir variables d'una en una

Ara la pregunta del dia. Afegirem variables en un ordre que té sentit intuïtiu:
primer les temperatures, després el sol i la pressió, i al final la pluja, el
vent i el calendari. A cada pas entrenarem un model nou i n'apuntarem els errors.

In [ ]:
resultats = []

for n in range(1, len(FEATURES) + 1):
    variables = FEATURES[:n]
    model = LinearRegression().fit(train[variables], train[OBJECTIU])

    resultats.append({
        "variables": n,
        "afegida": variables[-1],
        "MAE entrenament": mean_absolute_error(train[OBJECTIU], model.predict(train[variables])),
        "MAE validació": mean_absolute_error(valid[OBJECTIU], model.predict(valid[variables])),
        "R2 validació": r2_score(valid[OBJECTIU], model.predict(valid[variables])),
    })

resultats = pd.DataFrame(resultats)
resultats.round(3)

Llegeix la columna del MAE de validació de dalt a baix. Hi ha tres coses a veure:

**La primera variable fa gairebé tota la feina.** Passar de no tenir cap dada
(5,8 °C amb la mitjana) a tenir la màxima d'avui (1,61 °C) és un salt enorme.
Totes les altres variables juntes només baixen l'error unes quantes centèsimes més.

**Cada variable nova aporta menys que l'anterior.** És el que en economia
s'anomena *rendiments decreixents*, i passa gairebé sempre.

**Algunes no aporten res, o fins i tot empitjoren una mica.** La `tmed` deixa
l'error exactament igual: la temperatura mitjana és pràcticament el punt mig entre
la màxima i la mínima, i el model ja les té totes dues. **Una variable que repeteix
el que ja saps no afegeix informació.** Amb `presMax` i `prec` l'error puja una
mil·lèsima. Tampoc el mes i el dia de l'any mouen res. No és que el calendari no
tingui res a veure amb la temperatura, ja ho saps: és que, com veurem a la
secció 8, **una recta no el sap aprofitar**.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(resultats["variables"], resultats["MAE validació"], marker="o", label="model lineal")
plt.axhline(referencies.loc["demà igual que avui", "MAE (°C)"], color="gray", linestyle="--", label="demà igual que avui")
plt.xticks(resultats["variables"], resultats["afegida"], rotation=45)
plt.xlabel("Última variable afegida")
plt.ylabel("MAE de validació (°C)")
plt.legend()
plt.show()

Mira l'escala de l'eix vertical: tota la gràfica cap en menys d'una desena de
grau. La millora existeix, però és petita.

Fixa't també en una cosa de la taula: **els errors d'entrenament i de validació
són molt semblants**. El model s'equivoca pràcticament igual amb dies que ha vist
i amb dies que no. És el senyal que no està *memoritzant* res, i és el que
esperem d'una regressió lineal. Al NB 2.3 veurem un model que sí que memoritza, i
com es nota en aquestes dues columnes.

Afegir variables, a més, **té costos** encara que el número millori:

- Per fer una predicció, necessites **totes** les variables de cada dia. Si
  l'anemòmetre s'espatlla, el model de deu variables no pot predir.
- Un model amb deu coeficients és molt més difícil d'explicar que un amb un.
- Com més variables, més possibilitats que el model s'ajusti a casualitats de les
  dades d'entrenament. Ho veurem de prop al NB 2.3.

## 7. Llegir els coeficients quan n'hi ha molts

In [ ]:
model_tot = LinearRegression().fit(train[FEATURES], train[OBJECTIU])

coeficients = pd.Series(model_tot.coef_, index=FEATURES).round(3)
print(f"Ordenada a l'origen: {model_tot.intercept_:.2f}")
coeficients

Cada coeficient es llegeix gairebé com al NB 2.1, però amb una coletilla
important: **si la resta de variables no canvien**. Per exemple, el coeficient de
`sol` diu quant canvia la predicció per cada hora de sol més *d'un dia amb les
mateixes temperatures, la mateixa pressió i el mateix vent*.

I ara el parany. Seria temptador pensar que la variable amb el coeficient més gran
és la més important. **No es pot fer aquesta lectura**, per dos motius:

- **Les variables tenen unitats diferents.** Un grau, una hora de sol, un
  hectopascal de pressió i un mil·límetre de pluja no són comparables. Si
  mesuréssim la pluja en litres per hectàrea en lloc de mil·límetres, el seu
  coeficient canviaria de mida sense que el model canviés gens.
- **Les variables s'assemblen entre elles.** `tmax`, `tmin` i `tmed` pugen i baixen
  juntes, i `presMax` i `presMin` també. Quan dues variables diuen gairebé el
  mateix, el model es reparteix el pes entre elles d'una manera que pot semblar
  estranya. Mira les dues pressions: una té coeficient negatiu i l'altra
  positiu, i no té cap sentit físic que la pressió màxima faci baixar la
  temperatura i la mínima la faci pujar. És el model compensant una variable amb
  l'altra.

Com posar totes les variables a la mateixa escala és contingut de la **UT3**, i
com mesurar de veritat quina variable és més important, de la **UT5**.

## 8. Per què el calendari no ajuda?

Hem vist que `mes` i `dia_any` no milloren el model. Mirem-ho dibuixat.

In [ ]:
recta_dia = LinearRegression().fit(train[["dia_any"]], train[OBJECTIU])
dies = pd.DataFrame({"dia_any": np.arange(1, 367)})

plt.figure(figsize=(8, 4))
plt.scatter(train["dia_any"], train[OBJECTIU], alpha=0.15, s=8, label="dies d'entrenament")
plt.plot(dies["dia_any"], recta_dia.predict(dies), color="tab:red", linewidth=2, label="la millor recta")
plt.xlabel("Dia de l'any (1 = 1 de gener)")
plt.ylabel("Màxima de demà (°C)")
plt.legend()
plt.show()

La relació entre el dia de l'any i la temperatura és evident: fa fred al gener,
calor a l'agost i torna a fer fred al desembre. **Però no és una recta, és una
corba.** La millor recta que es pot traçar per aquest núvol puja una mica al llarg
de l'any, però no s'assembla gens a la forma de les dades: a l'agost es queda
molts graus per sota i al gener i al desembre, molts graus per sobre.

Això no vol dir que el dia de l'any no tingui informació. Vol dir que **el nostre
model només sap dibuixar rectes**. Hi ha dues sortides: donar-li al model una
manera de corbar-se, o donar-li la informació ja preparada perquè una recta la
pugui fer servir. La primera és el tema del NB 2.3; la segona, de la UT3.

## 9. Exercicis

**1. Un altre ordre.** Entrena un model només amb `sol`. Quin MAE de validació
té? Afegeix-hi `tmax`: quant millora? Dibuixa el núvol de punts de `sol` contra
`tmax_dema`: per què creus que les hores de sol, soles, prediuen tan malament la
temperatura si és evident que tenen alguna cosa a veure amb la calor?

**2. On s'equivoca més?** Amb el model de `tmax` sol (`model_1`), calcula el MAE
de validació **per a cada mes**. Quins mesos són els més difícils de predir?
Tens alguna hipòtesi de per què?
Pista: afegeix una columna d'error absolut a una còpia de `valid` i fes servir
`groupby("mes")`.

**3. El parany del percentatge.** Calcula la desviació percentual (NB 2.1) del
`model_1` sobre la validació. Després suma 273,15 a les temperatures reals i a
les predites (és a dir, passa-les a kelvins) i torna-la a calcular. Ha canviat el
MAE? I la desviació percentual? Què et diu això sobre quan es pot fer servir el
percentatge?

**4. Dos anys de validació.** Amb el model de totes les variables, calcula el MAE
per separat per al 2022 i per al 2023. Surten iguals? Què vol dir això sobre la
confiança que podem tenir en una sola xifra de validació?

**5. Per escrit.** Un company proposa: *"Només per curiositat, calculem el MAE del
model sobre el 2024 i el 2025. No canviarem res, només mirarem."* Explica en unes
quatre línies per què no s'ha de fer, fent servir el que has après a la secció 2.

## 10. Per al debat de classe

El model de deu variables s'equivoca unes set centèsimes de grau menys que el
model d'una sola variable.

- Si fossis el servei meteorològic, quin model posaries en marxa? Per què?
- Hi ha alguna situació on aquestes centèsimes podrien valer molts diners?
  (Pensa en empreses que depenen de la temperatura: energia, agricultura,
  turisme...)
- La persistència es calcula sense programar res i gairebé empata amb el nostre
  model. Vol dir que l'aprenentatge automàtic no serveix per predir la
  temperatura? Què li faltaria al nostre model per guanyar de manera clara?